In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. PARÂMETROS
# ============================================================

fs = 1000            # Frequência de amostragem (Hz)
N = 2000             # Número de amostras (2 s -> resolução de 0,5 Hz)
n = np.arange(N)
t = n / fs
f = np.fft.fftfreq(N, d=1 / fs)      # eixo de frequência (Hz), com negativas


# ============================================================
# 2. RESPOSTAS EM FREQUÊNCIA IDEAIS
# ============================================================
# Cada função devolve |H(f)| (0 ou 1) para um vetor de frequências f (Hz).
# Como o sinal é real, a máscara é simétrica em f (usa-se |f|).

def passa_baixas(f, fc):
    """Deixa passar |f| <= fc."""
    return (np.abs(f) <= fc).astype(float)


def passa_altas(f, fc):
    """Deixa passar |f| >= fc."""
    return (np.abs(f) >= fc).astype(float)


def passa_faixa(f, f1, f2):
    """Deixa passar f1 <= |f| <= f2."""
    return ((np.abs(f) >= f1) & (np.abs(f) <= f2)).astype(float)


def rejeita_faixa(f, f1, f2):
    """Rejeita f1 <= |f| <= f2."""
    return 1.0 - passa_faixa(f, f1, f2)


def filtrar_ideal(x, H):
    """Aplica a resposta ideal H (definida nos bins da FFT) ao sinal x."""
    return np.real(np.fft.ifft(np.fft.fft(x) * H))


def espectro_unilateral(x):
    """Magnitude unilateral (amplitude de cada senoide) e eixo em Hz."""
    X = np.fft.rfft(x) / len(x)
    mag = np.abs(X)
    mag[1:-1] *= 2
    return np.fft.rfftfreq(len(x), d=1 / fs), mag


def amplitude_em(x, f0):
    """Amplitude da componente de frequência f0 (Hz) no sinal x."""
    fr, mag = espectro_unilateral(x)
    return mag[np.argmin(np.abs(fr - f0))]


# ============================================================
# 3. CASOS DE APLICAÇÃO
# ============================================================
# Cada caso define: sinal desejado, componente indesejada, filtro e
# a interpretação física. As amplitudes e frequências são didáticas.

casos = []

# --- Passa-baixas: reduzir ruído de alta frequência ---
f_des, f_ind = 5, 200
casos.append(dict(
    nome="Passa-baixas",
    aplicacao="Reduzir ruído de alta frequência em um sinal lento",
    desejado=1.0 * np.sin(2 * np.pi * f_des * t),
    indesejado=0.5 * np.sin(2 * np.pi * f_ind * t),
    H=passa_baixas(f, 50),
    corte="fc = 50 Hz",
    f_des=f_des, f_ind=f_ind,
    interpretacao=("Elimina variações rápidas e preserva a tendência lenta "
                   "(suavização)."),
))

# --- Passa-altas: remover deriva (baixa frequência) ---
f_des, f_ind = 40, 0.5
casos.append(dict(
    nome="Passa-altas",
    aplicacao="Remover deriva de linha de base (nível DC lento)",
    desejado=1.0 * np.sin(2 * np.pi * f_des * t),
    indesejado=1.5 * np.sin(2 * np.pi * f_ind * t) + 0.8,   # deriva + DC
    H=passa_altas(f, 5),
    corte="fc = 5 Hz",
    f_des=f_des, f_ind=f_ind,
    interpretacao=("Remove o nível médio e as variações lentas; mantém "
                   "as variações rápidas."),
))

# --- Passa-faixa: selecionar uma faixa de interesse ---
f_des = 50
casos.append(dict(
    nome="Passa-faixa",
    aplicacao="Selecionar uma faixa de interesse entre outras componentes",
    desejado=1.0 * np.sin(2 * np.pi * f_des * t),
    indesejado=(0.8 * np.sin(2 * np.pi * 10 * t)
                + 0.6 * np.sin(2 * np.pi * 150 * t)),
    H=passa_faixa(f, 40, 60),
    corte="40 Hz a 60 Hz",
    f_des=f_des, f_ind=(10, 150),
    interpretacao=("Mantém só a faixa central; atenua tanto o que está "
                   "abaixo quanto o que está acima dela."),
))

# --- Rejeita-faixa: eliminar interferência da rede (60 Hz) ---
f_des, f_ind = 10, 60
casos.append(dict(
    nome="Rejeita-faixa",
    aplicacao="Eliminar interferência da rede elétrica (60 Hz)",
    desejado=1.0 * np.sin(2 * np.pi * f_des * t),
    indesejado=0.7 * np.sin(2 * np.pi * f_ind * t),
    H=rejeita_faixa(f, 55, 65),
    corte="rejeita 55 Hz a 65 Hz",
    f_des=f_des, f_ind=f_ind,
    interpretacao=("Remove uma faixa estreita e preserva o resto do "
                   "espectro (filtro notch)."),
))


# ============================================================
# 4. FILTRAGEM E MÉTRICAS
# ============================================================

for c in casos:
    c["x"] = c["desejado"] + c["indesejado"]
    c["y"] = filtrar_ideal(c["x"], c["H"])
    c["erro_max"] = np.max(np.abs(c["y"] - c["desejado"]))

print("ANÁLISE DOS FILTROS BÁSICOS")
print("=" * 78)
print(f"fs = {fs} Hz | N = {N} | resolução = {fs / N:.2f} Hz\n")

for c in casos:
    print(f"{c['nome']}  ({c['corte']})")
    print(f"  Aplicação: {c['aplicacao']}")
    # amplitude da componente desejada antes/depois
    a0, a1 = amplitude_em(c["x"], c["f_des"]), amplitude_em(c["y"], c["f_des"])
    print(f"  Componente desejada ({c['f_des']} Hz): "
          f"{a0:.3f} -> {a1:.3f}")
    inds = c["f_ind"] if isinstance(c["f_ind"], tuple) else (c["f_ind"],)
    for fi in inds:
        b0, b1 = amplitude_em(c["x"], fi), amplitude_em(c["y"], fi)
        print(f"  Componente indesejada ({fi} Hz): {b0:.3f} -> {b1:.3e}")
    print(f"  Erro máximo |y - desejado|: {c['erro_max']:.3e}")
    print(f"  Interpretação: {c['interpretacao']}\n")


# ============================================================
# 5. FIGURA 1: MAGNITUDE IDEAL DOS QUATRO FILTROS
# ============================================================

fig, eixos = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
f_plot = np.linspace(0, fs / 2, 5001)
ideais = [
    (passa_baixas(f_plot, 50), "Passa-baixas", [50]),
    (passa_altas(f_plot, 50), "Passa-altas", [50]),
    (passa_faixa(f_plot, 40, 60), "Passa-faixa", [40, 60]),
    (rejeita_faixa(f_plot, 40, 60), "Rejeita-faixa", [40, 60]),
]
# Os cortes da figura são ilustrativos (mesmos valores para comparar formas).
for ax, (H, nome, cortes) in zip(eixos.ravel(), ideais):
    ax.plot(f_plot, H, linewidth=2)
    ax.fill_between(f_plot, H, alpha=0.2, label="Banda passante")
    for fc in cortes:
        ax.axvline(fc, color="k", linestyle="--", linewidth=0.8)
    ax.set_title(nome)
    ax.set_ylim(-0.1, 1.2)
    ax.set_xlim(0, 200)
    ax.grid(True)
    ax.legend(loc="upper right")
for ax in eixos[1]:
    ax.set_xlabel("Frequência (Hz)")
for ax in eixos[:, 0]:
    ax.set_ylabel(r"$|H(f)|$")
plt.suptitle("Magnitude ideal dos filtros básicos (cortes ilustrativos)")
plt.tight_layout()
plt.show()


# ============================================================
# 6. FIGURAS 2-5: UM EXEMPLO DE APLICAÇÃO POR FILTRO
# ============================================================

t_ini, t_fim = 0, 0.4         # janela de tempo mostrada (s)
mask_t = (t >= t_ini) & (t < t_fim)

for c in casos:
    fig, eixos = plt.subplots(2, 2, figsize=(12, 7))
    fr, mag_x = espectro_unilateral(c["x"])
    _, mag_y = espectro_unilateral(c["y"])
    H_uni = np.interp(fr, f[: N // 2][::1], c["H"][: N // 2])

    # Tempo: entrada
    ax = eixos[0, 0]
    ax.plot(t[mask_t], c["x"][mask_t])
    ax.set_title("Entrada x[n] (desejado + indesejado)")
    ax.set_xlabel("Tempo (s)")
    ax.set_ylabel("Amplitude")
    ax.grid(True)

    # Tempo: saída x desejado
    ax = eixos[0, 1]
    ax.plot(t[mask_t], c["y"][mask_t], label="Saída y[n]")
    ax.plot(t[mask_t], c["desejado"][mask_t], "--", label="Sinal desejado")
    ax.set_title("Saída do filtro")
    ax.set_xlabel("Tempo (s)")
    ax.set_ylabel("Amplitude")
    ax.legend()
    ax.grid(True)

    # Frequência: espectro de entrada com H sobreposta
    ax = eixos[1, 0]
    ax.stem(fr, mag_x, basefmt=" ")
    ax.plot(fr, H_uni * mag_x.max(), "r--", label=r"$|H(f)|$ (escala)")
    ax.set_title("Espectro de entrada")
    ax.set_xlabel("Frequência (Hz)")
    ax.set_ylabel("Amplitude")
    ax.set_xlim(0, 250)
    ax.legend()
    ax.grid(True)

    # Frequência: espectro de saída
    ax = eixos[1, 1]
    ax.stem(fr, mag_y, basefmt=" ")
    ax.set_title("Espectro de saída")
    ax.set_xlabel("Frequência (Hz)")
    ax.set_ylabel("Amplitude")
    ax.set_xlim(0, 250)
    ax.grid(True)

    plt.suptitle(f"{c['nome']} ({c['corte']}): {c['aplicacao']}")
    plt.tight_layout()
    plt.show()

print("Simulação concluída!")